# YOLOv8n_run13s1｜run04 設定改用「不標停放自行車」資料（seed=1）

資料：v3 衍生資料 `v3_noparked`（由 Roboflow v3 移除 72 個停放自行車標註：train 43／valid 7／test 22；影像仍為 Train 664／Valid 135／Test 142 張），合併為 person／dynamic_obstacle。

**標註規則版本：新（2026/11/06 修訂）**：不標停放的自行車（含暫時停放；被牽行視為行駛中），由 `v3_raw` 移除 72 個 bicycle 標註（train 43／valid 7／test 22）而成 `v3_noparked`，Roboflow 未修改。本版的 test 與 run01～12 不是同一份評估，指標不可直接比較。

**變更（相對前一版）**

- 與 run04（無擴增） 設定相同（seed=1），僅資料改為移除停放自行車標註的衍生版本（train 43／valid 7／test 22 個 bicycle 標註被移除，影像不變）。
- 目的：依標註規則修訂（不標停放的自行車）重新評估；新舊 test 的指標不可直接比較。

共同設定：YOLOv8n、imgsz=320、epochs=100、patience=20、batch=32、seed=0、weight_decay=0.0005。本版 seed=1；與其他 seed 合併估計變異，差距在約 0.03 內不下結論。

In [ ]:
RUN = 'run13s1_v3noparked_baseline_seed1'

In [ ]:
# 安裝套件（版本與 run01~run03 相同）
!pip -q install ultralytics==8.4.183 roboflow

In [ ]:
# 環境偵測：Colab 或本地（Mac / Windows / Linux）皆可執行同一份程式
# - Colab：掛載 Drive，權重與結果直接寫入 Drive（斷線也不會遺失，可續跑）
# - 本地：資料放 ./work，結果放 ./results/<run>
import os, sys, torch
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content'
    OUT = f'/content/drive/MyDrive/capstone_yolo/{RUN}'
else:
    WORK = os.path.abspath('./work')
    OUT = os.path.abspath(f'./results/{RUN}')
os.makedirs(WORK, exist_ok=True); os.makedirs(OUT, exist_ok=True)
DEVICE = 0 if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
WORKERS = 2 if IN_COLAB else 4
print('環境：', 'Colab' if IN_COLAB else '本地', '｜device =', DEVICE, '｜結果目錄 =', OUT)

In [ ]:
# 【標註規則已修改（2026/11/06）】不標停放的自行車：與 run01～12 使用的標註不同，test 指標不可直接比較
# 使用本機衍生資料（由 colab_runs/make_noparked_dataset.py 從 v3_raw 建立，移除停放自行車標註；不下載、不耗 Roboflow 點數）
SRC = f'{WORK}/v3_noparked'
assert os.path.exists(f'{SRC}/data.yaml'), f'找不到 {SRC}；請先執行 colab_runs/make_noparked_dataset.py'
print('資料：', SRC)

In [ ]:
# 5 類合併為 2 類：person=0、其餘（bicycle/bus/car/motorcycle）=dynamic_obstacle=1
# 只改標籤檔，不耗 Roboflow 點數；遇到 polygon 時轉成外接矩形
import yaml, shutil, glob, collections
DST = f'{WORK}/data2_{RUN}'  # 每個 run 各用一份資料夾，避免擴增互相污染
names = yaml.safe_load(open(f'{SRC}/data.yaml'))['names']
if isinstance(names, dict): names = [names[k] for k in sorted(names)]
cmap = {i: (0 if n == 'person' else 1) for i, n in enumerate(names)}
print('原始類別順序：', names)
shutil.rmtree(DST, ignore_errors=True)
cnt = collections.defaultdict(collections.Counter)
for sp in ['train', 'valid', 'test']:
    os.makedirs(f'{DST}/{sp}/images', exist_ok=True); os.makedirs(f'{DST}/{sp}/labels', exist_ok=True)
    for im in glob.glob(f'{SRC}/{sp}/images/*'):
        shutil.copy2(im, f'{DST}/{sp}/images/')
    for lf in glob.glob(f'{SRC}/{sp}/labels/*.txt'):
        out = []
        for ln in open(lf).read().strip().splitlines():
            p = ln.split()
            c = cmap[int(p[0])]; v = list(map(float, p[1:]))
            if len(v) > 4:  # polygon -> bbox
                xs, ys = v[0::2], v[1::2]
                x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
                v = [(x0+x1)/2, (y0+y1)/2, x1-x0, y1-y0]
            out.append(' '.join([str(c)] + [f'{a:.6f}' for a in v])); cnt[sp][c] += 1
        open(f'{DST}/{sp}/labels/{os.path.basename(lf)}', 'w').write('\n'.join(out) + ('\n' if out else ''))
yaml.safe_dump({'path': DST, 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images',
                'names': {0: 'person', 1: 'dynamic_obstacle'}}, open(f'{DST}/data.yaml', 'w'))
for sp in cnt: print(sp, '影像', len(os.listdir(f'{DST}/{sp}/images')), '實例 person/dynamic_obstacle =', cnt[sp][0], cnt[sp][1])

In [ ]:
# 訓練：設定與 run02 相同（YOLOv8n、imgsz=320、epochs=100、patience=20、batch=32、seed=1、weight_decay=0.0005）
# 內建擴增全部關閉；本版不做任何擴增
from ultralytics import YOLO
LAST = f'{OUT}/runs/{RUN}/weights/last.pt'
resume = os.path.exists(LAST)  # 斷線或中斷後再執行這格，會從 last.pt 續跑（epoch 與學習率接續）
model = YOLO(LAST if resume else 'yolov8n.pt')
print('續跑' if resume else '從頭訓練')
model.train(**(dict(resume=True) if resume else dict(data=f'{DST}/data.yaml', imgsz=320, epochs=100, patience=20, batch=32, seed=1, deterministic=True,
            weight_decay=0.0005, workers=WORKERS, device=DEVICE, project=f'{OUT}/runs', name=RUN, exist_ok=True, save_period=10,
            hsv_h=0, hsv_s=0, hsv_v=0, degrees=0, translate=0, scale=0, shear=0, perspective=0,
            flipud=0, fliplr=0, mosaic=0, mixup=0, copy_paste=0, erasing=0)))

In [ ]:
# 以 best.pt 在 test 評估（test 只評估這一次；選模型看 valid）
best = YOLO(f'{OUT}/runs/{RUN}/weights/best.pt')
m = best.val(data=f'{DST}/data.yaml', split='test', imgsz=320, plots=True, device=DEVICE, project=f'{OUT}/runs', name=f'{RUN}_test', exist_ok=True)
print('mAP50', round(m.box.map50, 3), 'mAP50-95', round(m.box.map, 3), 'P', round(m.box.mp, 3), 'R', round(m.box.mr, 3))
# 逐類 AP：box.ap50 對應 ap_class_index（資料中實際出現的類別），不是全部類別名稱
for i, c in enumerate(m.box.ap_class_index):
    print(m.names[int(c)], 'AP50', round(float(m.box.ap50[i]), 3), '實例數', int(m.nt_per_class[int(c)]))

## 本版結論

| 指標 | valid | test |
|---|---|---|
| mAP50 | 0.682 | 0.744 |
| mAP50-95 | 0.384 | 0.413 |
| Precision | 0.866 | 0.819 |
| Recall | 0.562 | 0.671 |

| 類別（test，實例數） | AP50 | AP50-95 |
|---|---|---|
| person（137） | 0.826 | 0.407 |
| dynamic_obstacle（69） | 0.661 | 0.418 |


seed=1；best 在第 29 epoch，共 49 epoch。使用修訂後的標註規則（不標停放的自行車；test 的 dynamic_obstacle 為 69 個），與 run01～12 的 test 不可直接比較。

run13 三個 seed 的 test 平均：mAP50 0.733 ± 0.017、dynamic_obstacle AP50 0.656 ± 0.011、dynamic_obstacle recall 0.638 ± 0.025（見報告 v0.3 第 4.5 節）。